# Noisy GNLSE Propagation — Reparameterization Trick & FDT Balance

Tests `gnlse_solver_noisy.py`. Covers:

1. **Imports & physical parameters**
2. **Noiseless N=1 baseline** — saved to `figures/`
3. **Sigma sweep** — single trajectory at several noise levels
4. **Ensemble statistics** — mean ± std over realisations
5. **FDT balance demo** — pure ASE (energy grows) vs balanced loss (stationary E)
6. **Gradient test** — reparameterization via finite differences
7. **FD projection check**
8. **Loss-vs-sigma curve** — smooth, differentiable
9. **Coloured noise demo** — spectral shaping + step-size invariance

All figures saved under `figures/`.


In [ ]:


import os
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import time as time_mod

from gnlse_solver_noisy_new import (
    make_args,
    make_noise_samples,
    GNLSE3D_propagate,
    GNLSE3D_propagate_noisy,
    windowed_forward_noisy,
    windowed_grad_noisy,
    make_windowed_context_noisy,
    make_noise_filter,
    additive_noise_sigma,
    shot_noise_sigma,
    _prepare_propagation,
    _resolve_precision,
)

os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.90"
jax.config.update('jax_enable_x64', True)
print('JAX devices:', jax.devices())

os.makedirs('figures', exist_ok=True)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 9})
print('figures/ directory ready.')


In [ ]:
# ── Physical parameters (same as transfer_operator_1d_ist.ipynb) ──────────────
c0      = 2.998e8
lambda0 = 1064e-9
n_ref   = 1.453
n2      = 2.76e-20
# beta2 > 0 = anomalous in solver convention (see CLAUDE.md)
beta2   = +22e-27

omega0  = 2 * np.pi * c0 / lambda0
gamma   = n2 * omega0 / c0

T0      = 100e-15               # 100 fs sech width
L_D     = T0**2 / beta2
I_sol   = beta2 / (gamma * T0**2)   # N=1 soliton peak intensity [W/m^2]

print(f'gamma  = {gamma:.4e} m/W')
print(f'L_D    = {L_D*1e3:.4f} mm')
print(f'I_sol  = {I_sol:.3e} W/m^2  = peak |A|^2 for N=1')
print(f'sqrt(I_sol) = {np.sqrt(I_sol):.3e}  = peak |A| [sqrt(W/m^2)]')


In [ ]:
# ── Grid ──────────────────────────────────────────────────────────────────────
Nt   = 2048
Lt   = 10e-12        # 10 ps window
dt   = Lt / Nt
t    = np.arange(Nt) * dt
t_c  = Lt / 2
t_ps = t * 1e12

z_period = np.pi / 2 * L_D
Lz      = 3 * z_period
deltaZ  = L_D / 100
n_steps = int(round(Lz / deltaZ))
n_saves = 60

args = make_args(
    Nx=1, Ny=1, Nt=Nt,
    Lx=50e-6, Ly=50e-6,
    Lt=Lt, Lz=Lz,
    n_ref=n_ref, n2_val=n2,
    lambda0=lambda0, beta2_val=beta2,
    deltaZ=deltaZ,
    fr=0.0, sw=0, pml_thickness=0,
    precision='fp64', n_saves=n_saves,
)
print(f'Nt={Nt}, Lt={Lt*1e12:.0f} ps, T0={T0*1e15:.0f} fs')
print(f'Lz = {Lz*1e3:.3f} mm = {Lz/z_period:.0f} soliton periods')
print(f'Steps: {n_steps}, saves: {n_saves}')

# N=1 soliton IC
A_N1    = (np.sqrt(I_sol) / np.cosh((t - t_c) / T0)).astype(np.complex128)
A_N1_3d = A_N1[np.newaxis, np.newaxis, :]
E_sol_num = float(np.sum(np.abs(A_N1)**2) * dt)
z_mm    = np.linspace(0, Lz*1e3, n_saves)

# Noiseless baseline
print('\nPropagating noiseless N=1 soliton...')
t0 = time_mod.time()
res_clean = GNLSE3D_propagate_noisy(args, A_N1_3d, eps=None, noise_sigma=0.0, save_as_fp32=False)
print(f'Done in {time_mod.time()-t0:.1f}s')
field_clean = np.array(res_clean['field'][0, 0, :, :])   # (Nt, n_saves)
peak_clean  = np.max(np.abs(field_clean)**2, axis=0)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].pcolormesh(z_mm, t_ps, np.abs(field_clean)**2/I_sol,
                   shading='auto', cmap='inferno')
axes[0].set_ylim(t_ps[Nt//4], t_ps[3*Nt//4])
axes[0].set_xlabel('z (mm)'); axes[0].set_ylabel('t (ps)')
axes[0].set_title('Noiseless N=1 soliton: z-t map')

axes[1].plot(z_mm, peak_clean/I_sol)
axes[1].axhline(1.0, color='gray', ls='--')
axes[1].set_xlabel('z (mm)'); axes[1].set_ylabel('Peak |A|²/I_sol')
axes[1].set_title(f'Peak stability (σ = {np.std(peak_clean/I_sol)*100:.3f}%)')
axes[1].grid(alpha=0.3)

axes[2].plot(t_ps, np.abs(field_clean[:,0])**2/I_sol,  label='input')
axes[2].plot(t_ps, np.abs(field_clean[:,-1])**2/I_sol, '--', label='output')
axes[2].set_xlim(t_ps[Nt//4], t_ps[3*Nt//4])
axes[2].set_xlabel('t (ps)'); axes[2].legend()
axes[2].set_title('Input vs output'); axes[2].grid(alpha=0.3)

plt.suptitle('Noiseless baseline', fontsize=11)
plt.tight_layout()
plt.savefig('figures/01_noiseless_baseline.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/01_noiseless_baseline.png')


In [ ]:
# ── Single noisy trajectory at several σ values (pure ASE, loss_coeff=0) ──────
prep        = _prepare_propagation(args, A_N1_3d)
steps_total = prep['steps_total']
print(f'steps_total = {steps_total}')

sigma_fracs = [0.0, 0.01, 0.05, 0.20]
key0        = jax.random.PRNGKey(42)
eps         = make_noise_samples(key0, steps_total, 1, 1, Nt, dtype=jnp.complex128)
print(f'eps: mean |eps|² = {float(jnp.mean(jnp.abs(eps)**2)):.4f}  (expect 1.0)')

fig, axes = plt.subplots(len(sigma_fracs), 2, figsize=(13, 3.5*len(sigma_fracs)),
                         gridspec_kw=dict(wspace=0.35, hspace=0.5))

for row, frac in enumerate(sigma_fracs):
    sigma = additive_noise_sigma(A_N1_3d, frac)
    res   = GNLSE3D_propagate_noisy(args, A_N1_3d, eps, sigma,
                                    loss_coeff=0.0, save_as_fp32=False)
    f     = np.array(res['field'][0, 0, :, :])

    im = axes[row,0].pcolormesh(z_mm, t_ps, np.abs(f)**2/I_sol,
                                 shading='auto', cmap='inferno', vmin=0)
    axes[row,0].set_ylim(t_ps[Nt//4], t_ps[3*Nt//4])
    axes[row,0].set_xlabel('z (mm)'); axes[row,0].set_ylabel('t (ps)')
    axes[row,0].set_title(f'σ = {frac:.0%}√I_sol  (ASE only)', fontsize=9)
    plt.colorbar(im, ax=axes[row,0], label='|A|²/I_sol')

    axes[row,1].plot(t_ps, np.abs(f[:,0])**2/I_sol,  'C0', lw=1.5, label='input')
    axes[row,1].plot(t_ps, np.abs(f[:,-1])**2/I_sol, 'C1--', lw=1.5, label='output')
    axes[row,1].set_xlim(t_ps[Nt//4], t_ps[3*Nt//4])
    peak_out = np.max(np.abs(f[:,-1])**2)
    axes[row,1].set_title(f'Peak_out/I_sol = {peak_out/I_sol:.3f}', fontsize=9)
    axes[row,1].set_xlabel('t (ps)'); axes[row,1].legend(fontsize=8)
    axes[row,1].grid(alpha=0.3)

plt.suptitle('Pure ASE noise (loss_coeff=0): effect of σ on N=1 soliton (same ε realisation)', fontsize=10)
plt.tight_layout()
plt.savefig('figures/02_sigma_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/02_sigma_sweep.png')


In [ ]:
# ── Ensemble: mean ± std at fixed σ ──────────────────────────────────────────
sigma_frac = 0.05
sigma      = additive_noise_sigma(A_N1_3d, sigma_frac)
N_ens      = 20
key_ens    = jax.random.PRNGKey(1337)

I_out_ensemble = np.zeros((N_ens, Nt))
peak_ens       = np.zeros((N_ens, n_saves))

print(f'Running {N_ens} trajectories at σ={sigma_frac:.0%}√I_sol (ASE only)...')
t0 = time_mod.time()
for n in range(N_ens):
    key_ens, subkey = jax.random.split(key_ens)
    eps_n = make_noise_samples(subkey, steps_total, 1, 1, Nt, dtype=jnp.complex128)
    res_n = GNLSE3D_propagate_noisy(args, A_N1_3d, eps_n, sigma,
                                    loss_coeff=0.0, save_as_fp32=False)
    f_n   = np.array(res_n['field'][0, 0, :, :])
    I_out_ensemble[n] = np.abs(f_n[:, -1])**2
    peak_ens[n]       = np.max(np.abs(f_n)**2, axis=0)
print(f'{N_ens} trajectories in {time_mod.time()-t0:.1f}s')

I_mean = np.mean(I_out_ensemble, axis=0)
I_std  = np.std( I_out_ensemble, axis=0)
I_ref  = np.abs(field_clean[:, -1])**2

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ax = axes[0]
ax.fill_between(t_ps, (I_mean-I_std)/I_sol, (I_mean+I_std)/I_sol,
                alpha=0.35, color='C0', label='mean ± std')
ax.plot(t_ps, I_mean/I_sol, 'C0', lw=1.5)
ax.plot(t_ps, I_ref/I_sol,  'k--', lw=1.5, alpha=0.6, label='noiseless')
ax.set_xlim(t_ps[Nt//4], t_ps[3*Nt//4])
ax.set_xlabel('t (ps)'); ax.set_ylabel('|A|²/I_sol')
ax.set_title(f'Output: σ={sigma_frac:.0%}√I_sol, N={N_ens}')
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
for n in range(N_ens):
    ax.plot(z_mm, peak_ens[n]/I_sol, 'C0', alpha=0.25, lw=0.8)
ax.plot(z_mm, np.mean(peak_ens,axis=0)/I_sol, 'C1', lw=2, label='ensemble mean')
ax.plot(z_mm, peak_clean/I_sol, 'k--', lw=1.5, label='noiseless')
ax.axhline(1.0, color='gray', ls=':', alpha=0.5)
ax.set_xlabel('z (mm)'); ax.set_ylabel('Peak |A|²/I_sol')
ax.set_title('Peak intensity vs z')
ax.legend(); ax.grid(alpha=0.3)

ax = axes[2]
snr = I_mean**2 / (I_std**2 + 1e-30)
ax.semilogy(t_ps, snr)
ax.set_xlim(t_ps[Nt//4], t_ps[3*Nt//4])
ax.set_xlabel('t (ps)'); ax.set_ylabel('SNR = μ²/σ²')
ax.set_title('Signal-to-noise at output'); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('figures/03_ensemble.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/03_ensemble.png')


In [ ]:
# ── Fluctuation-Dissipation Balance ───────────────────────────────────────────
#
# Pure additive ASE (loss_coeff=0):
#   dE/dz = sigma² · Nt · dt   →  E(z) = E_sol + sigma² · Nt · dt · z  (linear growth)
#
# FDT-balanced (loss_coeff = sigma_frac²/2):
#   dA/dz = F(A) − loss_coeff·A + sigma·xi
#   Equilibrium:  ⟨|A|²⟩_eq = sigma²/(2·loss_coeff) = sigma_frac²·I_peak/(sigma_frac²) = I_peak
#   This keeps the energy stationary at E_sol; the noise temperature is fixed,
#   independent of propagation length.  (Using sigma²/2 instead of sigma_frac²/2
#   sets ⟨|A|²⟩_eq=1 ≪ I_sol and would zero out the field in a single step.)
#
# The FDT-balanced model is the right one for coherent structure discovery:
# it gives a fixed "temperature" so σ_k values are interpretable as structural
# stability at a well-defined noise level, not a z-dependent one.

sigma_fracs_fdt = [0.03, 0.07, 0.15]
N_ens_fdt = 10
key_fdt   = jax.random.PRNGKey(271828)

fig, axes = plt.subplots(len(sigma_fracs_fdt), 2,
                         figsize=(14, 4.2*len(sigma_fracs_fdt)),
                         gridspec_kw=dict(wspace=0.38, hspace=0.55))

for row, frac in enumerate(sigma_fracs_fdt):
    sigma  = additive_noise_sigma(A_N1_3d, frac)
    lc_bal = frac**2 / 2.0     # FDT balance: loss = sigma_frac²/2 → ⟨|A|²⟩_eq = I_sol
    # Note: sigma²/(2*I_sol) = (frac*sqrt(I_sol))²/(2*I_sol) = frac²/2
    # Using sigma²/2 (without /I_sol) would give ⟨|A|²⟩_eq=1, killing the soliton.

    e_ase = np.zeros((N_ens_fdt, n_saves))
    e_fdt = np.zeros((N_ens_fdt, n_saves))
    prof_ase, prof_fdt = [], []

    for n in range(N_ens_fdt):
        key_fdt, subkey = jax.random.split(key_fdt)
        eps_n = make_noise_samples(subkey, steps_total, 1, 1, Nt, dtype=jnp.complex128)

        r_ase = GNLSE3D_propagate_noisy(args, A_N1_3d, eps_n, sigma,
                                        loss_coeff=0.0, save_as_fp32=False)
        f_ase = np.array(r_ase['field'][0, 0, :, :])
        e_ase[n] = np.sum(np.abs(f_ase)**2, axis=0) * dt
        prof_ase.append(np.abs(f_ase[:, -1])**2)

        r_fdt = GNLSE3D_propagate_noisy(args, A_N1_3d, eps_n, sigma,
                                        loss_coeff=lc_bal, save_as_fp32=False)
        f_fdt = np.array(r_fdt['field'][0, 0, :, :])
        e_fdt[n] = np.sum(np.abs(f_fdt)**2, axis=0) * dt
        prof_fdt.append(np.abs(f_fdt[:, -1])**2)

    # ── Left: energy vs z ─────────────────────────────────────────────────
    ax = axes[row, 0]
    for n in range(N_ens_fdt):
        ax.plot(z_mm, e_ase[n]/E_sol_num, color='C1', alpha=0.18, lw=0.7)
        ax.plot(z_mm, e_fdt[n]/E_sol_num, color='C0', alpha=0.18, lw=0.7)
    ax.plot(z_mm, e_ase.mean(0)/E_sol_num, 'C1', lw=2.2, label='ASE (loss=0)')
    ax.plot(z_mm, e_fdt.mean(0)/E_sol_num, 'C0', lw=2.2, label='FDT (loss=σ²/2)')
    # Analytical ASE drift
    z_arr    = np.array(res_clean['field'][0,0,:,:]).shape   # just need n_saves
    z_arr    = np.linspace(0, Lz, n_saves)
    e_theory = (E_sol_num + float(sigma)**2 * Nt * dt * z_arr) / E_sol_num
    ax.plot(z_mm, e_theory, 'C1--', lw=1.3, alpha=0.8, label='theory: E₀+σ²·Nₜ·dt·z')
    ax.axhline(1.0, color='k', ls=':', lw=1.0, alpha=0.4)
    ax.set_xlabel('z (mm)'); ax.set_ylabel('E(z) / E_sol')
    ax.set_title(f'σ = {frac:.0%}√I_sol  |  loss_coeff = {lc_bal:.2e} m⁻¹')
    ax.legend(fontsize=7.5); ax.grid(alpha=0.3)

    # ── Right: output intensity (mean ± std) ──────────────────────────────
    ax = axes[row, 1]
    I_ase = np.array(prof_ase) / I_sol
    I_fdt = np.array(prof_fdt) / I_sol
    I_ref = np.abs(field_clean[:, -1])**2 / I_sol
    ax.fill_between(t_ps, I_ase.mean(0)-I_ase.std(0), I_ase.mean(0)+I_ase.std(0),
                    color='C1', alpha=0.25)
    ax.fill_between(t_ps, I_fdt.mean(0)-I_fdt.std(0), I_fdt.mean(0)+I_fdt.std(0),
                    color='C0', alpha=0.25)
    ax.plot(t_ps, I_ase.mean(0), 'C1', lw=1.8, label='ASE mean')
    ax.plot(t_ps, I_fdt.mean(0), 'C0', lw=1.8, label='FDT mean')
    ax.plot(t_ps, I_ref,         'k--', lw=1.2, alpha=0.55, label='noiseless')
    ax.set_xlim(t_ps[Nt//4], t_ps[3*Nt//4])
    ax.set_xlabel('t (ps)'); ax.set_ylabel('|A|²/I_sol')
    ax.set_title(f'Output profile  ({N_ens_fdt} realisations, mean ± std)')
    ax.legend(fontsize=7.5); ax.grid(alpha=0.3)

plt.suptitle(
    'Fluctuation-Dissipation Balance\n'
    'ASE: energy grows linearly with z.  FDT (loss=σ²/2): E stationary at E_sol.',
    fontsize=11)
plt.tight_layout()
plt.savefig('figures/04_fdt_balance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/04_fdt_balance.png')
print()
print('FDT summary:')
for frac in sigma_fracs_fdt:
    sigma = additive_noise_sigma(A_N1_3d, frac)
    lc    = frac**2 / 2   # sigma_frac²/2, independent of field scale
    L_damp = 1.0 / (2*lc) if lc > 0 else float('inf')
    print(f'  σ_frac={frac:.0%}: sigma={sigma:.3e}, loss_coeff={lc:.4f} m⁻¹, '
          f'L_damp={L_damp*1e3:.1f} mm  ({L_damp/L_D:.1f} × L_D)')


In [ ]:
# ── Gradient test: verify reparameterization via finite differences ───────────
#
# Loss: L = sum |A_final(t)|^2  (total output energy)
# Test: analytical dL/dA0 from windowed_grad_noisy vs finite differences.
#
# We use a short propagation (1 soliton period) and few windows for speed.

# Short args for fast FD test
Lz_test = 0.5 * z_period     # half soliton period
args_test = make_args(
    Nx=1, Ny=1, Nt=Nt,
    Lx=50e-6, Ly=50e-6,
    Lt=Lt, Lz=Lz_test,
    n_ref=n_ref, n2_val=n2,
    lambda0=lambda0, beta2_val=beta2,
    deltaZ=deltaZ,
    fr=0.0, sw=0,
    pml_thickness=0,
    precision='fp64',
    n_saves=10,
)

prep_test   = _prepare_propagation(args_test, A_N1_3d)
steps_test  = prep_test['steps_total']
print(f'Test steps: {steps_test}')

key_test = jax.random.PRNGKey(999)
eps_test = make_noise_samples(key_test, steps_test, 1, 1, Nt, dtype=jnp.complex128)
sigma_test = 0.03 * np.sqrt(I_sol)

# Build context once
ctx_test = make_windowed_context_noisy(args_test, Nt, loss_coeff=0.0)
# loss_coeff=0 for gradient test: FDT damping is differentiable but we test the base case

# Loss in k-space: L = ||A_final_kwo||^2  (proportional to Parseval energy)
def loss_fn(field_kwo_final):
    return jnp.sum(jnp.abs(field_kwo_final)**2).real

n_win = 4
print(f'Computing analytical gradient (n_windows={n_win})...')
res_grad = windowed_grad_noisy(
    loss_fn, args_test, A_N1_3d, eps_test, sigma_test,
    n_windows=n_win, ctx=ctx_test,
)
grad_analytical = np.array(res_grad['grad'])   # (1, 1, Nt) complex kwo-space
print(f'Loss = {float(res_grad["loss"]):.6e}')
print(f'Grad norm = {np.linalg.norm(grad_analytical):.6e}')
print(f'fwd: {res_grad["fwd_seconds"]:.1f}s  bwd: {res_grad["bwd_seconds"]:.1f}s')


In [ ]:
# ── Finite-difference check on a random projection ────────────────────────────
# We test dL/dA0_kwo along a random direction v.
# JAX complex gradient convention: for real f, FD = Re(g · v)  (no conjugate, no factor 2).

key_fd = jax.random.PRNGKey(2025)
v_rand = np.array(jax.random.normal(key_fd, A_N1_3d.shape, dtype=jnp.float64), dtype=np.complex128)
v_rand /= np.linalg.norm(v_rand)

def compute_loss_at(A0_perturbed):
    fwd = windowed_forward_noisy(
        args_test, A0_perturbed, eps_test, sigma_test,
        n_windows=n_win, ctx=ctx_test, save_as_fp32=False
    )
    kwo = jnp.fft.fftn(fwd['field_final'], axes=(0,1,2))
    return float(loss_fn(kwo))

h_vals = [1e-1, 1e-2, 5e-3]
# v in k-space (FFT of v_xyt)
v_kwo = np.fft.fftn(v_rand, axes=(0,1,2))
# Analytical directional derivative: Re(g · v_kwo) — JAX convention for complex grads
directional_analytical = np.real(np.sum(grad_analytical * v_kwo))

print(f'Analytical directional derivative: {directional_analytical:.6e}')
print()

for h in h_vals:
    Lp = compute_loss_at(A_N1_3d + h * v_rand)
    Lm = compute_loss_at(A_N1_3d - h * v_rand)
    fd = (Lp - Lm) / (2 * h)
    rel_err = abs(fd - directional_analytical) / (abs(directional_analytical) + 1e-30)
    print(f'h={h:.0e}:  FD = {fd:.6e}   rel_err = {rel_err:.2e}')


In [ ]:
# ── Gradient w.r.t. noise_sigma via finite differences ───────────────────────
# Demonstrate that the reparameterization makes dL/d(sigma) well-defined.
# We perturb sigma directly and check the numerical derivative.

def loss_vs_sigma(sigma_val):
    """Scalar loss as a function of sigma (all else fixed)."""
    fwd = windowed_forward_noisy(
        args_test, A_N1_3d, eps_test, float(sigma_val),
        n_windows=n_win, ctx=ctx_test, save_as_fp32=False
    )
    kwo = jnp.fft.fftn(fwd['field_final'], axes=(0,1,2))
    return float(loss_fn(kwo))

sigma_vals  = np.linspace(0.0, 0.1 * np.sqrt(I_sol), 30)
loss_curve  = [loss_vs_sigma(s) for s in sigma_vals]
print('Loss vs sigma curve computed.')

# Finite-difference slope at sigma_test
delta_s = sigma_test * 0.01
dL_ds_fd = (loss_vs_sigma(sigma_test + delta_s) - loss_vs_sigma(sigma_test - delta_s)) / (2*delta_s)

# Analytical slope from the reparameterization:
# d/d(sigma) L(A0 + sigma*eps) = <grad_A L, eps_sum>
# where eps_sum = sum over steps of the propagated eps contribution.
# This is NOT trivially computable without a second backward pass;
# we just verify the curve is smooth and the FD slope is finite.
print(f'dL/d(sigma) at sigma_test (FD) = {dL_ds_fd:.4e}  [finite, gradient is usable]')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(sigma_vals / np.sqrt(I_sol) * 100, loss_curve, 'C0o-', ms=3)
axes[0].axvline(sigma_test/np.sqrt(I_sol)*100, color='C1', ls='--', label='σ_test')
axes[0].set_xlabel('σ / √I_sol  (%)')
axes[0].set_ylabel('Loss = ||A_final||²_k')
axes[0].set_title('Loss is smooth and differentiable w.r.t. σ (reparameterization)')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Overlay several noise-level trajectories final profiles
key_disp = jax.random.PRNGKey(77)
eps_disp = make_noise_samples(key_disp, steps_test, 1, 1, Nt, dtype=jnp.complex128)
frac_vals = [0.0, 0.02, 0.05, 0.10]
for frac in frac_vals:
    fwd = windowed_forward_noisy(
        args_test, A_N1_3d, eps_disp, frac*np.sqrt(I_sol),
        n_windows=n_win, ctx=ctx_test, save_as_fp32=False
    )
    Af = np.array(fwd['field_final'][0, 0, :])
    axes[1].plot(t_ps, np.abs(Af)**2/I_sol, label=f'σ={frac:.0%}√I_sol')
axes[1].set_xlim(t_ps[Nt//4], t_ps[3*Nt//4])
axes[1].set_xlabel('t (ps)'); axes[1].set_ylabel('|A_final|²/I_sol')
axes[1].set_title('Output profiles at varying σ (windowed_forward_noisy)')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('figures/05_loss_vs_sigma.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Coloured noise demo ──────────────────────────────────────────────────────
# Demonstrates:
#  1. make_noise_filter shapes the noise power spectrum as expected
#  2. sqrt(deltaZ) fix: accumulated noise power is step-size invariant

from gnlse_solver_noisy import make_noise_filter

# ── 1. Build and visualise filter shapes ─────────────────────────────────────
freq_axis = np.fft.fftfreq(Nt, dt) * 1e-12   # THz, fftfreq ordering
freq_ps   = np.fft.fftshift(freq_axis)         # shift for plotting

filters = {
    'white (no filter)':         None,
    'Gaussian 1 THz':   make_noise_filter(Nt, dt, bandwidth_hz=1e12,  filter_type='gaussian'),
    'Gaussian 5 THz':   make_noise_filter(Nt, dt, bandwidth_hz=5e12,  filter_type='gaussian'),
    'Lorentzian 2 THz': make_noise_filter(Nt, dt, bandwidth_hz=2e12,  filter_type='lorentzian'),
    'Rect ±3 THz':      make_noise_filter(Nt, dt, bandwidth_hz=3e12,  filter_type='rect'),
}

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for label, H in filters.items():
    if H is None:
        H_plot = np.ones(Nt)
    else:
        H_plot = np.array(H)
    axes[0].plot(freq_ps, np.fft.fftshift(H_plot), label=label)

axes[0].set_xlabel('Frequency (THz)'); axes[0].set_ylabel('H(f)  [normalised]')
axes[0].set_title('Colour filter shapes'); axes[0].legend(fontsize=8)
axes[0].axhline(1, color='gray', ls=':', alpha=0.4); axes[0].grid(alpha=0.3)

# ── 2. Measure output noise power spectrum for each filter ────────────────────
# Propagate noiseless once to get the clean field at every save point.
# Then subtract from noisy runs to isolate the noise contribution.
sigma_demo  = 0.04 * np.sqrt(I_sol)    # 4% of sqrt(I_sol)
key_demo    = jax.random.PRNGKey(5555)
eps_demo    = make_noise_samples(key_demo, steps_total, 1, 1, Nt, dtype=jnp.complex128)

for label, H in filters.items():
    res = GNLSE3D_propagate_noisy(
        args, A_N1_3d, eps_demo, sigma_demo,
        loss_coeff=0.0, noise_filter_w=H, save_as_fp32=False,
    )
    A_out = np.array(res['field'][0, 0, :, -1])
    A_ref = field_clean[:, -1]
    delta_A   = A_out - A_ref                        # noise contribution at output
    psd       = np.abs(np.fft.fftshift(np.fft.fft(delta_A)))**2
    axes[1].plot(freq_ps, 10*np.log10(psd / psd.max() + 1e-12), label=label)

axes[1].set_xlabel('Frequency (THz)'); axes[1].set_ylabel('Noise PSD (dB, normalised)')
axes[1].set_title('Output noise spectrum — matches filter shape'); axes[1].set_ylim(-60, 5)
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

plt.suptitle('make_noise_filter: spectral shaping of per-step noise', fontsize=11)
plt.tight_layout()
plt.savefig('figures/06_coloured_noise_filters.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/06_coloured_noise_filters.png')

# ── 3. Step-size invariance check ────────────────────────────────────────────
# Same sigma, halved deltaZ: total noise power at output should be ~equal.
print('\n── Step-size invariance (sqrt(deltaZ) fix) ──')
sigma_inv = 0.03 * np.sqrt(I_sol)

for dZ_factor, label in [(1, 'nominal deltaZ'), (2, 'deltaZ / 2')]:
    dZ_here  = deltaZ / dZ_factor
    args_inv = make_args(
        Nx=1, Ny=1, Nt=Nt, Lx=50e-6, Ly=50e-6,
        Lt=Lt, Lz=Lz, n_ref=n_ref, n2_val=n2,
        lambda0=lambda0, beta2_val=beta2,
        deltaZ=dZ_here, fr=0.0, sw=0, pml_thickness=0,
        precision='fp64', n_saves=n_saves,
    )
    prep_inv  = _prepare_propagation(args_inv, A_N1_3d)
    N_inv     = prep_inv['steps_total']
    key_inv   = jax.random.PRNGKey(7777)
    eps_inv   = make_noise_samples(key_inv, N_inv, 1, 1, Nt, dtype=jnp.complex128)

    # Run 5 independent realisations and average noise energy at output
    energies = []
    for trial in range(5):
        key_inv, sub = jax.random.split(key_inv)
        eps_t = make_noise_samples(sub, N_inv, 1, 1, Nt, dtype=jnp.complex128)
        res   = GNLSE3D_propagate_noisy(args_inv, A_N1_3d, eps_t, sigma_inv,
                                         loss_coeff=0.0, save_as_fp32=False)
        A_out = np.array(res['field'][0, 0, :, -1])
        delta = A_out - field_clean[:, -1]
        energies.append(float(np.sum(np.abs(delta)**2) * dt))

    print(f'  {label:20s}: mean noise energy = {np.mean(energies):.3e} ± {np.std(energies):.1e}')

print('\nValues should agree within statistical fluctuations.')
print('Without sqrt(deltaZ) fix, halving deltaZ would double the noise energy.')
print('\nStep-size invariance check complete. See terminal output above.')


In [ ]:
# ── Shot noise demo ────────────────────────────────────────────────────────────
# Compare additive (ASE) noise vs shot noise at the same nominal sigma_frac.
# Shot noise: injection amplitude ∝ sqrt(|A|) → zero in dark regions.
#
# Use additive_noise_sigma / shot_noise_sigma helpers to compute the correctly-
# dimensioned noise_sigma from the initial field and a dimensionless fraction:
#   Additive:  sigma ∝ √I_peak   →  equal noise at all points per √m of propagation
#   Shot:      sigma ∝ I_peak^¼  →  noise ∝ √|A|, zero in dark regions
# Both give the same noise-to-signal ratio at the field peak.
#
# FDT balance: loss_coeff = sigma_frac²/2 → ⟨|A|²⟩_eq = I_ref (for additive case;
# shot noise equilibrium is approximately the same for small perturbations).
#
# Variables inherited from earlier cells:
#   steps_total  — number of z-steps (from _prepare_propagation in cell 4)
#   A_N1_3d      — (1, 1, Nt) initial field
#   I_sol        — soliton peak intensity [W/m^2]
#   t_ps         — time axis in picoseconds (0 … Lt ps)
#   t_c          — pulse centre time [s]

sigma_frac = 0.05
sigma_add  = additive_noise_sigma(A_N1_3d, sigma_frac)  # 5% × √I_peak  [A/√m]
sigma_shot = shot_noise_sigma(A_N1_3d, sigma_frac)        # 5% × I_peak^¼ [A^½/√m]
#   → equal noise-to-signal amplitude at the field peak for both formulas

# FDT balance: loss = sigma_frac²/2 so that ⟨|A|²⟩_eq = I_sol.
loss_shot  = sigma_frac**2 / 2              # = 0.00125 m⁻¹, L_damp ≈ 400 m
N_ens      = 10                             # ensemble size

peak_add  = []
peak_shot = []

for seed in range(N_ens):
    key = jax.random.PRNGKey(seed + 200)
    eps = make_noise_samples(key, steps_total, 1, 1, Nt, dtype=jnp.complex128)

    # Additive noise (use_shot_noise=False, the default)
    out_add = GNLSE3D_propagate_noisy(
        args, A_N1_3d, eps, noise_sigma=sigma_add,
        loss_coeff=loss_shot, use_shot_noise=False,
    )
    A_add = np.asarray(out_add['field'][0, 0, :, -1])   # last snapshot, shape (Nt,)
    peak_add.append(np.max(np.abs(A_add)**2))

    # Shot noise (use_shot_noise=True); sigma has different units — see header.
    out_shot = GNLSE3D_propagate_noisy(
        args, A_N1_3d, eps, noise_sigma=sigma_shot,
        loss_coeff=loss_shot, use_shot_noise=True,
    )
    A_shot = np.asarray(out_shot['field'][0, 0, :, -1])
    peak_shot.append(np.max(np.abs(A_shot)**2))

peak_add  = np.array(peak_add)
peak_shot = np.array(peak_shot)

# ── Profile plot (seed 200) + ensemble peak statistics ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

key0 = jax.random.PRNGKey(200)
eps0 = make_noise_samples(key0, steps_total, 1, 1, Nt, dtype=jnp.complex128)

out_add0  = GNLSE3D_propagate_noisy(args, A_N1_3d, eps0, noise_sigma=sigma_add,
                                     loss_coeff=loss_shot, use_shot_noise=False)
out_shot0 = GNLSE3D_propagate_noisy(args, A_N1_3d, eps0, noise_sigma=sigma_shot,
                                     loss_coeff=loss_shot, use_shot_noise=True)

ic_prof    = np.abs(A_N1_3d[0, 0, :])**2 / I_sol
A_add_prof = np.abs(np.asarray(out_add0['field'][0, 0, :, -1]))**2 / I_sol
A_shot_prof= np.abs(np.asarray(out_shot0['field'][0, 0, :, -1]))**2 / I_sol

# Zoom window: ±0.5 ps around pulse centre
tc_ps   = t_c * 1e12
xlim_ps = (tc_ps - 0.5, tc_ps + 0.5)

ax = axes[0]
ax.plot(t_ps, ic_prof,     'k--', lw=1.2, label='Input')
ax.plot(t_ps, A_add_prof,  'C0',  lw=1.5, label=f'Additive (σ_frac={sigma_frac*100:.0f}%)')
ax.plot(t_ps, A_shot_prof, 'C1',  lw=1.5, label='Shot noise (same σ_frac)')
ax.set_xlabel('Time (ps)')
ax.set_ylabel('Intensity / I_sol')
ax.set_title('Output profile: additive vs shot noise')
ax.set_xlim(*xlim_ps)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

ax = axes[1]
ax.bar(['Additive', 'Shot'], [peak_add.mean()/I_sol, peak_shot.mean()/I_sol],
       yerr=[peak_add.std()/I_sol, peak_shot.std()/I_sol],
       color=['C0','C1'], capsize=6, width=0.4)
ax.axhline(1.0, color='k', lw=1, ls='--', label='Noiseless peak')
ax.set_ylabel('Peak intensity / I_sol')
ax.set_title(f'Peak intensity statistics ({N_ens} realisations)')
ax.legend(fontsize=8)

plt.suptitle('Shot noise vs additive (ASE) noise, FDT-balanced', fontsize=10)
plt.tight_layout()
plt.savefig('figures/07_shot_noise_demo.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Additive   peak: {peak_add.mean()/I_sol:.3f} ± {peak_add.std()/I_sol:.3f} I_sol')
print(f'Shot noise peak: {peak_shot.mean()/I_sol:.3f} ± {peak_shot.std()/I_sol:.3f} I_sol')
print('Shot noise is suppressed in low-intensity wings (sqrt(|A|)→0 where field→0).')


## 2+1D Transverse Propagation: Profiles and Animations

The cells below simulate a Gaussian beam propagating in a step-index waveguide
using a 2+1D (x, y, z) spatial NLSE (CW: `Nt=1`, `beta2=0`).
This isolates transverse diffraction and Kerr self-focusing from temporal effects.

Three noise regimes are compared using `make_xy_z_animation`, `make_1d_z_animation`,
and `make_transverse_vs_z_vs_I_plot` from `gnlse_visualizations_prototype`:

| Label | `loss_coeff` | `use_shot_noise` | Physics |
|-------|-------------|------------------|---------|
| Additive ASE | 0 | False | Uniform noise; energy grows with z |
| Additive FDT | σ_frac²/2 | False | Uniform noise; energy stationary |
| Shot FDT | σ_frac²/2 | True | Noise ∝ √\|A\|; dark cladding preserved |

**σ_frac is set large (×5) so cladding noise is clearly visible on a linear colormap.**

In [ ]:


# ── 2+1D setup: transverse grid, step-index waveguide, Gaussian beam ──────────
from gnlse_medium import make_space, make_supergauss_index
from gnlse_visualizations_prototype import (
    make_xy_z_animation,
    make_1d_z_animation,
    make_transverse_vs_z_vs_I_plot,
)
from IPython.display import Image, display

# Transverse grid — Nt=1 → pure 2+1D CW spatial NLSE (no temporal dispersion)
Nx_2d = Ny_2d = 400
Lx_2d = Ly_2d = 100e-6     # 50 μm transverse window
Nt_2d = 1                  # single time bin; temporal FFTs become trivial
Lt_2d = 1e-12              # nominal time window (irrelevant for Nt=1)

x_2d = np.linspace(-Lx_2d/2, Lx_2d/2, Nx_2d)
y_2d = np.linspace(-Ly_2d/2, Ly_2d/2, Ny_2d)
X_2d, Y_2d = np.meshgrid(x_2d, y_2d, indexing='ij')  # (Nx, Ny)

# Step-index waveguide (super-Gaussian index profile)
r_core_2d  = 5e-6             # 5 μm core radius
n_core_2d  = 1.463            # Δn = 0.010 above cladding
n_clad_2d  = n_ref            # 1.453 silica
n_xy_2d    = np.array(make_supergauss_index(
    X_2d, Y_2d, n_core_2d, n_clad_2d, r_core_2d, m=8))
# n_xy_2d shape (Nx, Ny); make_args tiles to (Nx, Ny, Nt=1) automatically

# Propagation parameters — 10 Rayleigh lengths so we see guided evolution
z_R_2d    = np.pi * n_clad_2d * r_core_2d**2 / lambda0  # Rayleigh length in medium
Lz_2d     = 10 * z_R_2d
deltaZ_2d = Lz_2d / 1000
n_saves_2d = 200

# Initial Gaussian beam matched to core waist
w0_2d  = r_core_2d
I_2d   = I_sol                 # same peak intensity as 1D soliton section
A_beam = np.sqrt(I_2d) * np.exp(-(X_2d**2 + Y_2d**2) / (2 * w0_2d**2))
A_2d_3d = A_beam[:, :, np.newaxis].astype(np.complex128)  # (Nx, Ny, 1)

args_2d = make_args(
    Nx=Nx_2d, Ny=Ny_2d, Nt=Nt_2d,
    Lx=Lx_2d, Ly=Ly_2d, Lt=Lt_2d, Lz=Lz_2d,
    n_ref=n_core_2d, n2_val=n2, n_xyomega=n_xy_2d,
    lambda0=lambda0, beta2_val=0.0, deltaZ=deltaZ_2d,
    fr=0.0, sw=0, pml_thickness=100, pml_eta=0.0,
    precision='fp64', n_saves=n_saves_2d,
)
z_save_2d = np.array(args_2d['save_at'])   # (n_saves_2d,) z positions [m]
prep_2d   = _prepare_propagation(args_2d, A_2d_3d)
steps_2d  = prep_2d['steps_total']
dy_2d     = float(prep_2d['dy'])
ext_um    = np.array([-Lx_2d/2, Lx_2d/2, -Ly_2d/2, Ly_2d/2]) * 1e6  # plot extent [μm]

print(f'z_R = {z_R_2d*1e6:.0f} μm,  Lz = {Lz_2d*1e3:.2f} mm = {Lz_2d/z_R_2d:.0f}×z_R')
print(f'{steps_2d} z-steps, {n_saves_2d} save points')
L_NL_2d = 1 / (gamma * I_2d)
print(f'L_NL = {L_NL_2d*1e3:.1f} mm  (Lz/L_NL = {Lz_2d/L_NL_2d:.3f} → weakly nonlinear)')

# ── Initial beam: intensity + phase ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
im0 = axes[0].imshow(np.abs(A_beam).T**2 / I_2d,
                     origin='lower', extent=ext_um, cmap='hot', vmin=0, vmax=1)
axes[0].contour(x_2d*1e6, y_2d*1e6, n_xy_2d.T,
                levels=[n_clad_2d + 0.005], colors='cyan', linewidths=1.5, linestyles='--')
plt.colorbar(im0, ax=axes[0], label='|A|² / I₂D')
axes[0].set_title('Initial intensity  (cyan = core boundary)')
axes[0].set_xlabel('x (μm)'); axes[0].set_ylabel('y (μm)')

_ph_cmap = plt.cm.hsv.copy(); _ph_cmap.set_bad('lightgray')
_amp_beam = np.abs(A_beam)
_ph_beam  = np.where(_amp_beam >= 0.05*_amp_beam.max(), np.angle(A_beam), np.nan)
im1 = axes[1].imshow(_ph_beam.T,
                     origin='lower', extent=ext_um, cmap=_ph_cmap, vmin=-np.pi, vmax=np.pi)
plt.colorbar(im1, ax=axes[1], label='phase (rad)')
axes[1].set_title('Initial phase')
axes[1].set_xlabel('x (μm)'); axes[1].set_ylabel('y (μm)')

plt.suptitle('2+1D: Gaussian beam in step-index waveguide (input)', fontsize=10)
plt.tight_layout()
plt.savefig('figures/08_2d_initial_beam.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Noiseless propagation ──────────────────────────────────────────────────────
print('\nPropagating 2+1D noiseless...')
t0 = time_mod.time()
out_2d_clean = GNLSE3D_propagate_noisy(
    args_2d, A_2d_3d, eps=None, noise_sigma=0.0, save_as_fp32=False)
print(f'Done in {time_mod.time()-t0:.1f}s')
field4d_clean = np.array(out_2d_clean['field'])   # (64, 64, 1, 50)

# make_transverse_vs_z_vs_I_plot expects out dict with 'dx' and 'dy'
out_2d_clean_viz = dict(**out_2d_clean, dy=dy_2d)

# ── Static: x vs z intensity map ──────────────────────────────────────────────
fig_xvz, ax_xvz, *_ = make_transverse_vs_z_vs_I_plot(
    out_2d_clean_viz, args_2d,
    axis='x', mode='time_integrated', reduce='centerline',
    normalize=True, cmap='hot',
    title='Noiseless: intensity  |  x–z map (y = 0 centerline)',
    figsize=(9, 4),
)
plt.savefig('figures/09_2d_noiseless_xvz.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Static: output transverse intensity + phase ────────────────────────────────
I_out_clean  = np.abs(field4d_clean[:, :, 0, -1])**2   # (Nx, Ny)
_f_out_clean = field4d_clean[:, :, 0, -1]
_ph_cmap = plt.cm.hsv.copy(); _ph_cmap.set_bad('lightgray')
_amp_oc   = np.abs(_f_out_clean)
ph_out_clean = np.where(_amp_oc >= 0.05*_amp_oc.max(), np.angle(_f_out_clean), np.nan)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
im0 = axes[0].imshow(I_out_clean.T / I_2d,
                     origin='lower', extent=ext_um, cmap='hot', vmin=0, vmax=1)
axes[0].contour(x_2d*1e6, y_2d*1e6, n_xy_2d.T,
                levels=[n_clad_2d+0.005], colors='cyan', linewidths=1.5, linestyles='--')
plt.colorbar(im0, ax=axes[0], label='|A|² / I₂D')
axes[0].set_title(f'Output intensity  (z = {Lz_2d*1e3:.2f} mm)')
axes[0].set_xlabel('x (μm)'); axes[0].set_ylabel('y (μm)')

im1 = axes[1].imshow(ph_out_clean.T,
                     origin='lower', extent=ext_um, cmap=_ph_cmap, vmin=-np.pi, vmax=np.pi)
plt.colorbar(im1, ax=axes[1], label='phase (rad)')
axes[1].set_title('Output phase')
axes[1].set_xlabel('x (μm)'); axes[1].set_ylabel('y (μm)')

plt.suptitle('2+1D Noiseless: output transverse profile', fontsize=10)
plt.tight_layout()
plt.savefig('figures/10_2d_noiseless_output.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Animations ─────────────────────────────────────────────────────────────────
print('Creating noiseless animations...')
# Transverse intensity as beam propagates in z
make_xy_z_animation(
    field4d_clean, t_index=0,
    x=x_2d*1e6, y=y_2d*1e6, z=z_save_2d*1e6,
    quantity='intensity', norm='global', fps=12,
    filename='figures/anim_2d_noiseless_intensity.gif', dpi=100,
)
# Phase pattern as beam propagates in z
make_xy_z_animation(
    field4d_clean, t_index=0,
    x=x_2d*1e6, y=y_2d*1e6, z=z_save_2d*1e6,
    quantity='phase', norm='per_frame', fps=12,
    filename='figures/anim_2d_noiseless_phase.gif', dpi=100,
)
# x-slice intensity (centerline) along z
make_1d_z_animation(
    field4d_clean, coord_axis='x', reduce_method='centerline',
    coord=x_2d*1e6, z=z_save_2d*1e6,
    quantity='intensity', norm='global', fps=12,
    filename='figures/anim_2d_noiseless_xslice.gif', dpi=100,
)
print('Done. Displaying animations:')
for fname in ['figures/anim_2d_noiseless_intensity.gif',
              'figures/anim_2d_noiseless_phase.gif',
              'figures/anim_2d_noiseless_xslice.gif']:
    print(f'  {fname}')
    display(Image(fname))


In [ ]:
# ── 2+1D noise comparison: additive ASE, additive FDT, shot noise FDT ─────────
#
# sigma_frac_2d = 5 (500%) is intentionally large so that the cladding noise
# floor (≈ sigma_frac² × I_2d × Lz ≈ 2.7% I_2d) is visible on a linear scale.
# The FDT damping length L_damp = 1/(2×loss) = 1/(2×12.5) = 40 mm >> Lz = 1 mm,
# so it barely acts within the simulation — its effect shows up over ensembles.
#
# Key observable: shot noise leaves the dark cladding CLEAN (noise ∝ √|A| → 0
# where |A|→0), while additive noise lights up the cladding uniformly.

sigma_frac_2d = 5.0
sigma_2d      = additive_noise_sigma(A_2d_3d, sigma_frac_2d)  # [A/√m]
sigma_2d_shot = shot_noise_sigma(A_2d_3d, sigma_frac_2d)          # [A^½/√m]
loss_fdt_2d   = sigma_frac_2d**2 / 2                  # FDT balance → ⟨|A|²⟩_eq = I_2d

key_2d  = jax.random.PRNGKey(888)
eps_2d  = make_noise_samples(key_2d, steps_2d, Nx_2d, Ny_2d, Nt_2d,
                              dtype=jnp.complex128)

print('Running 2+1D propagations...')
out_2d_ase  = GNLSE3D_propagate_noisy(
    args_2d, A_2d_3d, eps_2d, noise_sigma=sigma_2d,
    loss_coeff=0.0, use_shot_noise=False, save_as_fp32=False)
print('  ASE done')
out_2d_fdt  = GNLSE3D_propagate_noisy(
    args_2d, A_2d_3d, eps_2d, noise_sigma=sigma_2d,
    loss_coeff=loss_fdt_2d, use_shot_noise=False, save_as_fp32=False)
print('  FDT done')
out_2d_shot = GNLSE3D_propagate_noisy(
    args_2d, A_2d_3d, eps_2d, noise_sigma=sigma_2d_shot,
    loss_coeff=loss_fdt_2d, use_shot_noise=True, save_as_fp32=False)
print('  Shot done')

field4d_ase  = np.array(out_2d_ase['field'])
field4d_fdt  = np.array(out_2d_fdt['field'])
field4d_shot = np.array(out_2d_shot['field'])

# Extract output slice (last z save, Nt=0)
cases = [
    ('Noiseless',      field4d_clean),
    ('Additive ASE',   field4d_ase),
    ('Additive FDT',   field4d_fdt),
    ('Shot FDT',       field4d_shot),
]
I_outs  = {lbl: np.abs(f[:, :, 0, -1])**2     for lbl, f in cases}
_ph_cmap = plt.cm.hsv.copy(); _ph_cmap.set_bad('lightgray')
def _masked_phase(f2d):
    amp = np.abs(f2d); return np.where(amp >= 0.05*amp.max(), np.angle(f2d), np.nan)
ph_outs = {lbl: _masked_phase(f[:, :, 0, -1]) for lbl, f in cases}

# Expected cladding noise floor for additive (theoretical)
noise_floor = sigma_2d**2 * Lz_2d   # ≈ sigma_frac²×I_2d×Lz
zoom_vmax   = max(0.01 * I_2d, 3 * noise_floor)  # show ~3× noise floor
print(f'Cladding noise floor ≈ {noise_floor/I_2d*100:.1f}% I₂D')
print(f'Zoomed colormap vmax = {zoom_vmax/I_2d*100:.1f}% I₂D')

# ── 4-panel comparison: full intensity range ───────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for col, (lbl, _) in enumerate(cases):
    # Top row: intensity (full range)
    ax = axes[0, col]
    im = ax.imshow(I_outs[lbl].T / I_2d,
                   origin='lower', extent=ext_um, cmap='hot', vmin=0, vmax=1)
    ax.contour(x_2d*1e6, y_2d*1e6, n_xy_2d.T,
               levels=[n_clad_2d+0.005], colors='cyan', linewidths=1.0, linestyles='--')
    ax.set_title(lbl, fontsize=9)
    ax.set_xlabel('x (μm)')
    if col == 0:
        ax.set_ylabel('y (μm)\n[full range]')
        plt.colorbar(im, ax=ax, label='|A|²/I₂D')

    # Bottom row: zoomed intensity (shows cladding noise)
    ax = axes[1, col]
    im2 = ax.imshow(I_outs[lbl].T / I_2d,
                    origin='lower', extent=ext_um, cmap='hot',
                    vmin=0, vmax=zoom_vmax/I_2d)
    ax.contour(x_2d*1e6, y_2d*1e6, n_xy_2d.T,
               levels=[n_clad_2d+0.005], colors='cyan', linewidths=1.0, linestyles='--')
    ax.set_xlabel('x (μm)')
    if col == 0:
        ax.set_ylabel(f'y (μm)\n[zoomed ≤{zoom_vmax/I_2d*100:.0f}% I₂D]')
        plt.colorbar(im2, ax=ax, label='|A|²/I₂D')

plt.suptitle(
    f'2+1D output intensity  |  σ = {sigma_frac_2d:.0f}×√I₂D  |  '
    f'z = {Lz_2d*1e3:.2f} mm\n'
    f'Top: full range.  Bottom: zoomed to cladding — shot noise leaves dark cladding clean.',
    fontsize=9)
plt.tight_layout()
plt.savefig('figures/11_2d_noise_intensity_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# ── 4-panel comparison: phase ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
for col, (lbl, _) in enumerate(cases):
    ax = axes[col]
    im = ax.imshow(ph_outs[lbl].T,
                   origin='lower', extent=ext_um, cmap=_ph_cmap, vmin=-np.pi, vmax=np.pi)
    ax.contour(x_2d*1e6, y_2d*1e6, n_xy_2d.T,
               levels=[n_clad_2d+0.005], colors='white', linewidths=1.0, linestyles='--')
    ax.set_title(lbl, fontsize=9)
    ax.set_xlabel('x (μm)')
    if col == 0:
        ax.set_ylabel('y (μm)')
    plt.colorbar(im, ax=ax, label='phase (rad)')
plt.suptitle(
    'Phase comparison — additive noise randomises phase everywhere (incl. cladding);\n'
    'shot noise leaves cladding phase undisturbed (near-zero field → no noise).',
    fontsize=9)
plt.tight_layout()
plt.savefig('figures/12_2d_noise_phase_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# ── x-slice profile comparison ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
colors_2d = ['k', 'C1', 'C0', 'C2']
for (lbl, _), col in zip(cases, colors_2d):
    xslice = I_outs[lbl][:, Ny_2d//2]   # y = 0 centerline
    axes[0].plot(x_2d*1e6, xslice/I_2d, color=col, lw=1.8, label=lbl)
    axes[1].semilogy(x_2d*1e6, np.maximum(xslice, 1e-20)/I_2d, color=col, lw=1.8, label=lbl)
for ax in axes:
    ax.axvline(-r_core_2d*1e6, color='gray', ls=':', lw=1.2, label='core edge')
    ax.axvline( r_core_2d*1e6, color='gray', ls=':', lw=1.2)
    ax.set_xlabel('x (μm)')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
axes[0].set_ylabel('|A|² / I₂D  (linear)')
axes[1].set_ylabel('|A|² / I₂D  (log)')
axes[0].set_title('x-slice at y=0  (linear scale)')
axes[1].set_title('x-slice at y=0  (log scale — reveals cladding noise)')
plt.suptitle(f'Output x-slice  |  z = {Lz_2d*1e3:.2f} mm  |  σ = {sigma_frac_2d:.0f}×√I₂D', fontsize=9)
plt.tight_layout()
plt.savefig('figures/13_2d_xslice_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Shot noise: cladding noise floor ≪ additive noise floor (log scale shows this clearly).')

# ── x vs z heatmaps for each case ────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (lbl, f4d), out_2d in zip(
    axes, cases,
    [out_2d_clean, out_2d_ase, out_2d_fdt, out_2d_shot]
):
    out_viz = dict(**out_2d, dy=dy_2d)
    make_transverse_vs_z_vs_I_plot(
        out_viz, args_2d,
        axis='x', mode='time_integrated', reduce='centerline',
        normalize=True, cmap='hot',
        title=lbl, figsize=(4, 4),
    )
    plt.savefig(f'figures/14_2d_xvz_{lbl.replace(" ","_").lower()}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

# ── Animations for each case ───────────────────────────────────────────────────
print('\nCreating animations (this may take a minute)...')
anim_cases = [
    ('Noiseless',    field4d_clean, 'noiseless'),
    ('Additive ASE', field4d_ase,   'ase'),
    ('Additive FDT', field4d_fdt,   'fdt'),
    ('Shot FDT',     field4d_shot,  'shot'),
]
for lbl, f4d, tag in anim_cases:
    fin  = f'figures/anim_2d_{tag}_intensity.gif'
    fph  = f'figures/anim_2d_{tag}_phase.gif'
    fxsl = f'figures/anim_2d_{tag}_xslice.gif'

    print(f'  {lbl}: intensity xy-z animation...')
    make_xy_z_animation(
        f4d, t_index=0,
        x=x_2d*1e6, y=y_2d*1e6, z=z_save_2d*1e6,
        quantity='intensity', norm='global', fps=12,
        filename=fin, dpi=100,
    )
    print(f'  {lbl}: phase xy-z animation...')
    make_xy_z_animation(
        f4d, t_index=0,
        x=x_2d*1e6, y=y_2d*1e6, z=z_save_2d*1e6,
        quantity='phase', norm='per_frame', fps=12,
        filename=fph, dpi=100,
    )
    print(f'  {lbl}: x-slice animation...')
    make_1d_z_animation(
        f4d, coord_axis='x', reduce_method='centerline',
        coord=x_2d*1e6, z=z_save_2d*1e6,
        quantity='intensity', norm='global', fps=12,
        filename=fxsl, dpi=100,
    )
    print(f'  Displaying: {lbl}')
    display(Image(fin))
    display(Image(fph))
    display(Image(fxsl))

print('\nAll 2+1D animations complete.')


In [ ]:
print('=== Summary ===')
print(f'Noiseless soliton peak stability : ±{np.std(peak_clean/I_sol)*100:.3f}%')
print(f'Gradient norm (kwo space)        : {np.linalg.norm(grad_analytical):.4e}')
print(f'FD gradient check                : see rel_err printed above')
print(f'dL/d(sigma) at 3% sigma (FD)     : {dL_ds_fd:.4e}')
print()
print('FDT balance (loss_coeff = sigma_frac²/2):')
for frac in [0.03, 0.07, 0.15]:
    sigma = additive_noise_sigma(A_N1_3d, frac)
    lc    = frac**2 / 2   # = sigma_frac²/2, independent of field scale
    print(f'  σ_frac={frac:.0%}: sigma={sigma:.3e}, loss_coeff={lc:.4f} m⁻¹')
print()
print('Figures saved:')
for fname in sorted(os.listdir('figures')):
    if fname.endswith('.png'):
        path = os.path.join('figures', fname)
        size = os.path.getsize(path)
        print(f'  {path}  ({size//1024} KB)')
